# Fase 2d: Incrementalidad (PSM + Uplift Modeling)

> **⚠️ Limitación causal:** El treatment (canjear) y el outcome (gasto post) se miden en la misma ventana temporal (6 meses post-t0). Esto significa que los resultados miden **asociación**, no causalidad. Para inferencia causal verdadera se necesitaría una ventana SEL separada del POST, o un experimento aleatorio.

**Objetivo:** Estimar el efecto causal del canje sobre el gasto posterior, mejorando la metodología actual de lift por decil.

**Metodología existente (producción):**
- 3 periodos: PREV → SEL → POST
- Matching por decil de gasto previo
- Lift = (gasto_canjeador - gasto_potencial) / gasto_potencial

**Nuestra mejora:**
1. **PSM** — Propensity Score Matching sobre múltiples covariables (no solo decil de gasto)
2. **Uplift Modeling** — T-Learner / X-Learner para estimar efecto individual (CATE)
3. **Revenue Decomposition** — base + incremental

**Treatment:** T = canjeó en los 6 meses post-t0 (y ≥ 1)  
**Outcome:** Y = gasto total en los 6 meses post-t0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print("Imports OK")

In [ ]:
# ── Carga de datos ─────────────────────────────────────────────────────────
USE_MOCK = True

if USE_MOCK:
    import duckdb
    with open("../../fase1/test_mock_local.py") as f:
        code = f.read().replace("con.close()", "# con.close()")
    _globals = {}
    exec(code, _globals)
    con = _globals['con']
    df = con.execute("SELECT * FROM customer_snapshot").df()
    
    # ── Calcular outcome: gasto post-t0 desde transacciones ──────────────
    spending_post = con.execute("""
        WITH snap AS (
            SELECT DISTINCT cust_id, t0, t0 + INTERVAL '6 months' AS t0_end
            FROM customer_snapshot
        )
        SELECT s.cust_id, s.t0,
               COALESCE(SUM(t.tran_amt), 0) AS spending_post_6m,
               COUNT(t.tran_date) AS txn_count_post_6m
        FROM snap s
        LEFT JOIN mock_transaction_entity t
            ON t.cust_id = s.cust_id
            AND t.tran_date >= s.t0
            AND t.tran_date < s.t0_end
        GROUP BY s.cust_id, s.t0
    """).df()
    
    con.close()
    print(f"Snapshot: {df.shape}")
    print(f"Spending post: {spending_post.shape}")
else:
    from google.cloud import bigquery
    client = bigquery.Client(project="my-gcp-project")
    df = client.query("SELECT * FROM `my-gcp-project.loyalty_intelligence.customer_snapshot`").to_dataframe()
    # En prod: calcular spending_post desde tablon de transacciones
    spending_post = pd.DataFrame()  # placeholder

df['t0'] = pd.to_datetime(df['t0'])
spending_post['t0'] = pd.to_datetime(spending_post['t0'])

# Merge outcome al snapshot
df = df.merge(spending_post, on=['cust_id', 't0'], how='left')
df['spending_post_6m'] = df['spending_post_6m'].fillna(0)

# Treatment: canjeó post-t0
df['treatment'] = (df['y'] >= 1).astype(int)

print(f"\nShape final: {df.shape}")
print(f"Treatment rate: {df['treatment'].mean():.1%}")
print(f"t0s: {sorted(df['t0'].dt.strftime('%Y-%m').unique())}")

In [ ]:
# ── Definir covariables pre-treatment para PSM ──────────────────────────
# Solo features observables PRE-t0 que afectan tanto el tratamiento como el outcome
PSM_FEATURES = [
    'frequency_monthly_avg',
    'monetary_monthly_avg',
    'redeem_rate',
    'retailer_entropy',
    'pct_redeem_digital',
    'earn_velocity_90',
    'days_since_last_activity',
    'points_pressure',
    'stock_points_at_t0',
    'redeem_count_pre',
    'frequency_total',
    'monetary_total',
    'tenure_months',
]

# Filtrar features disponibles
PSM_FEATURES = [f for f in PSM_FEATURES if f in df.columns]

# Calcular retailer_entropy si falta (basado en frecuencia, no gasto)
if 'retailer_entropy' not in df.columns:
    freqs = df[['freq_store_a', 'freq_store_b', 'freq_store_c', 'freq_store_d', 'freq_store_e']].values
    tot = np.where(freqs.sum(axis=1, keepdims=True) == 0, 1, freqs.sum(axis=1, keepdims=True))
    p = freqs / tot
    df['retailer_entropy'] = -(p * np.where(p > 0, np.log(p), 0)).sum(axis=1)

# Preparar X
X_psm = df[PSM_FEATURES].fillna(0).copy()
T = df['treatment'].values
Y = df['spending_post_6m'].values

print(f"Covariables PSM: {len(PSM_FEATURES)}")
print(f"Treated: {T.sum():,} ({T.mean():.1%}) | Control: {(1-T).sum():,.0f} ({(1-T).mean():.1%})")
print(f"Outcome (spending_post_6m): mean={Y.mean():,.0f}, median={np.median(Y):,.0f}")

## 1. Comparación Naive (sesgada)

In [ ]:
# ── Diferencia naive: canjeadores vs no canjeadores ──────────────────────
# Esto tiene selection bias: quienes canjean ya eran diferentes ANTES
treated_spending = Y[T == 1]
control_spending = Y[T == 0]

naive_diff = treated_spending.mean() - control_spending.mean()
naive_lift = naive_diff / max(control_spending.mean(), 1)

print("COMPARACION NAIVE (SESGADA — solo referencia)")
print("="*60)
print(f"Gasto post-6m canjeadores:    ${treated_spending.mean():>12,.0f}  (n={len(treated_spending):,})")
print(f"Gasto post-6m no canjeadores: ${control_spending.mean():>12,.0f}  (n={len(control_spending):,})")
print(f"Diferencia naive:             ${naive_diff:>12,.0f}")
print(f"Lift naive:                    {naive_lift:>11.1%}")
print(f"\n⚠ Este lift sobreestima el efecto real (selection bias)")

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(control_spending, bins=30, alpha=0.6, label='No canjeó', color='gray')
axes[0].hist(treated_spending, bins=30, alpha=0.6, label='Canjeó', color='steelblue')
axes[0].set_xlabel('Gasto post-6m')
axes[0].set_title('Distribución de Gasto Post-t0')
axes[0].legend()

# Diferencia de medias de covariables (pre-matching)
smd_pre = []
for feat in PSM_FEATURES:
    t_mean = X_psm.loc[T == 1, feat].mean()
    c_mean = X_psm.loc[T == 0, feat].mean()
    pooled_std = np.sqrt((X_psm.loc[T == 1, feat].std()**2 + X_psm.loc[T == 0, feat].std()**2) / 2)
    smd = (t_mean - c_mean) / max(pooled_std, 1e-9)
    smd_pre.append({'feature': feat, 'SMD': smd})

smd_df = pd.DataFrame(smd_pre).sort_values('SMD', key=abs, ascending=True)
colors = ['red' if abs(s) > 0.1 else 'green' for s in smd_df['SMD']]
axes[1].barh(smd_df['feature'], smd_df['SMD'].abs(), color=colors)
axes[1].axvline(0.1, color='red', linestyle='--', alpha=0.5, label='|SMD|=0.1')
axes[1].set_xlabel('|Standardized Mean Difference|')
axes[1].set_title('Balance Pre-Matching (rojo = desbalanceado)')
axes[1].legend()

plt.tight_layout()
plt.show()

n_unbalanced = sum(1 for s in smd_df['SMD'] if abs(s) > 0.1)
print(f"\nCovariables desbalanceadas (|SMD|>0.1): {n_unbalanced}/{len(PSM_FEATURES)}")

## 2. Propensity Score Matching (PSM)

In [ ]:
# ── Estimar propensity score: P(T=1 | X) — Cross-fitted (5-fold) ─────────
# Cross-fitting prevents overfitting leak: each observation's propensity
# is predicted by a model that never saw that observation during training.
# shuffle=False to respect temporal ordering of t0 periods.
from sklearn.model_selection import StratifiedKFold

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_psm)

# Cross-fitted propensity scores
propensity = np.zeros(len(X_scaled))
skf = StratifiedKFold(n_splits=5, shuffle=False)
fold_aucs = []
for fold_train, fold_val in skf.split(X_scaled, T):
    fold_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    fold_model.fit(X_scaled[fold_train], T[fold_train])
    propensity[fold_val] = fold_model.predict_proba(X_scaled[fold_val])[:, 1]
    fold_aucs.append(roc_auc_score(T[fold_val], propensity[fold_val]))

# Final model on all data (for use in other notebooks)
ps_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
ps_model.fit(X_scaled, T)

df['propensity_score'] = propensity

auc_ps = roc_auc_score(T, propensity)
print(f"Propensity Score model AUC (cross-fitted): {auc_ps:.4f}")
print(f"  Fold AUCs: {[f'{a:.4f}' for a in fold_aucs]}")
print(f"  (AUC ~0.5 = tratamiento casi random → no hay sesgo)")
print(f"  (AUC ~1.0 = tratamiento muy predecible → sesgo fuerte)")

# Distribución del propensity score por grupo
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(propensity[T == 0], bins=50, alpha=0.6, density=True, label='Control (no canjeó)', color='gray')
ax.hist(propensity[T == 1], bins=50, alpha=0.6, density=True, label='Treated (canjeó)', color='steelblue')
ax.set_xlabel('Propensity Score P(T=1|X)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución del Propensity Score (Cross-Fitted)')
ax.legend()
plt.tight_layout()
plt.show()

# Common support: rango donde ambos grupos tienen densidad
ps_treated = propensity[T == 1]
ps_control = propensity[T == 0]
common_min = max(ps_treated.min(), ps_control.min())
common_max = min(ps_treated.max(), ps_control.max())
print(f"\nCommon support: [{common_min:.4f}, {common_max:.4f}]")

in_support = (propensity >= common_min) & (propensity <= common_max)
print(f"En common support: {in_support.sum():,}/{len(propensity):,} ({in_support.mean():.1%})")

In [ ]:
# ── Matching: nearest neighbor sobre propensity score ────────────────────
# Filtrar a common support
df_cs = df[in_support].copy()
T_cs = df_cs['treatment'].values
Y_cs = df_cs['spending_post_6m'].values
ps_cs = df_cs['propensity_score'].values

# Para cada treated, encontrar el control mas cercano en propensity
idx_treated = np.where(T_cs == 1)[0]
idx_control = np.where(T_cs == 0)[0]

nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(ps_cs[idx_control].reshape(-1, 1))

distances, neighbors = nn.kneighbors(ps_cs[idx_treated].reshape(-1, 1))
matched_control_idx = idx_control[neighbors.flatten()]

# Caliper: excluir matches con distancia > 0.05
CALIPER = 0.05
good_match = distances.flatten() <= CALIPER

matched_treated_idx = idx_treated[good_match]
matched_control_idx = matched_control_idx[good_match]

print(f"Matching con caliper={CALIPER}:")
print(f"  Treated matcheados: {len(matched_treated_idx):,}/{len(idx_treated):,} ({len(matched_treated_idx)/len(idx_treated):.1%})")
print(f"  Distancia media: {distances[good_match].mean():.4f}")
print(f"  Distancia max:   {distances[good_match].max():.4f}")

In [ ]:
# ── Verificar balance POST-matching ──────────────────────────────────────
X_cs = df_cs[PSM_FEATURES].fillna(0).values

smd_post = []
for i, feat in enumerate(PSM_FEATURES):
    t_mean = X_cs[matched_treated_idx, i].mean()
    c_mean = X_cs[matched_control_idx, i].mean()
    pooled_std = np.sqrt((X_cs[matched_treated_idx, i].std()**2 + X_cs[matched_control_idx, i].std()**2) / 2)
    smd = (t_mean - c_mean) / max(pooled_std, 1e-9)
    smd_post.append({'feature': feat, 'SMD_pre': smd_df.set_index('feature').loc[feat, 'SMD'], 'SMD_post': smd})

balance_df = pd.DataFrame(smd_post)

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(balance_df))
ax.scatter(balance_df['SMD_pre'].abs(), y_pos, marker='o', s=80, color='red', label='Pre-matching', zorder=3)
ax.scatter(balance_df['SMD_post'].abs(), y_pos, marker='s', s=80, color='green', label='Post-matching', zorder=3)
for i in y_pos:
    ax.plot([balance_df['SMD_pre'].abs().iloc[i], balance_df['SMD_post'].abs().iloc[i]], [i, i],
            color='gray', alpha=0.5)
ax.axvline(0.1, color='red', linestyle='--', alpha=0.5, label='|SMD|=0.1')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(balance_df['feature'])
ax.set_xlabel('|Standardized Mean Difference|')
ax.set_title('Balance de Covariables: Pre vs Post Matching')
ax.legend()
plt.tight_layout()
plt.show()

n_balanced = (balance_df['SMD_post'].abs() <= 0.1).sum()
print(f"Covariables balanceadas post-matching (|SMD|≤0.1): {n_balanced}/{len(PSM_FEATURES)}")
print(f"\nBalance detallado:")
print(balance_df.round(4).to_string(index=False))

In [ ]:
# ── Estimar ATT (Average Treatment Effect on Treated) ────────────────────
Y_treated_matched = Y_cs[matched_treated_idx]
Y_control_matched = Y_cs[matched_control_idx]

att = Y_treated_matched.mean() - Y_control_matched.mean()
att_lift = att / max(Y_control_matched.mean(), 1)

# Bootstrap CI para ATT
np.random.seed(42)
n_boot = 1000
att_boots = []
n_matched = len(matched_treated_idx)
for _ in range(n_boot):
    idx = np.random.choice(n_matched, size=n_matched, replace=True)
    boot_att = Y_treated_matched[idx].mean() - Y_control_matched[idx].mean()
    att_boots.append(boot_att)

att_ci_lo = np.percentile(att_boots, 2.5)
att_ci_hi = np.percentile(att_boots, 97.5)

print("RESULTADOS PSM")
print("="*60)
print(f"Gasto post-6m (matched treated):  ${Y_treated_matched.mean():>12,.0f}")
print(f"Gasto post-6m (matched control):  ${Y_control_matched.mean():>12,.0f}")
print(f"ATT (efecto causal estimado):      ${att:>12,.0f}")
print(f"ATT 95% CI:                        [${att_ci_lo:>10,.0f}, ${att_ci_hi:>10,.0f}]")
print(f"Lift PSM:                           {att_lift:>11.1%}")
print(f"\nComparacion:")
print(f"  Lift naive:  {naive_lift:.1%}")
print(f"  Lift PSM:    {att_lift:.1%}")
print(f"  Sesgo corregido: {abs(naive_lift - att_lift)/max(abs(naive_lift),1e-9)*100:.0f}% del naive era sesgo")

In [ ]:
# ── Tests estadísticos sobre el ATT ──────────────────────────────────────
from scipy import stats

# 1) t-test pareado (matched treated vs matched control)
t_stat, t_pval = stats.ttest_rel(Y_treated_matched, Y_control_matched)

# 2) Wilcoxon signed-rank (no-parametrico, pareado)
w_stat, w_pval = stats.wilcoxon(Y_treated_matched - Y_control_matched)

# 3) Permutation test: bajo H0 el treatment label es intercambiable
np.random.seed(42)
n_perm = 5000
perm_diffs = []
combined = np.stack([Y_treated_matched, Y_control_matched], axis=1)
for _ in range(n_perm):
    swaps = np.random.binomial(1, 0.5, size=len(combined))
    y_t_perm = np.where(swaps, combined[:, 0], combined[:, 1])
    y_c_perm = np.where(swaps, combined[:, 1], combined[:, 0])
    perm_diffs.append(y_t_perm.mean() - y_c_perm.mean())

perm_diffs = np.array(perm_diffs)
perm_pval = (np.abs(perm_diffs) >= abs(att)).mean()

print("TESTS ESTADISTICOS SOBRE ATT")
print("="*60)
print(f"ATT = ${att:,.0f}")
print(f"\n{'Test':<30} {'Estadístico':<15} {'p-value':<10} {'Sig (α=0.05)'}")
print("-"*70)
print(f"{'t-test pareado':<30} {t_stat:<15.3f} {t_pval:<10.4f} {'✓' if t_pval < 0.05 else '✗'}")
print(f"{'Wilcoxon signed-rank':<30} {w_stat:<15,.0f} {w_pval:<10.4f} {'✓' if w_pval < 0.05 else '✗'}")
print(f"{'Permutation test (n=5000)':<30} {'—':<15} {perm_pval:<10.4f} {'✓' if perm_pval < 0.05 else '✗'}")

# Efecto estandarizado (Cohen's d)
pooled_std_att = np.sqrt((Y_treated_matched.std()**2 + Y_control_matched.std()**2) / 2)
cohens_d = att / pooled_std_att
print(f"\nCohen's d: {cohens_d:.3f} ({'pequeño' if abs(cohens_d)<0.2 else 'mediano' if abs(cohens_d)<0.5 else 'grande'})")

# Visualizar permutation distribution
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(perm_diffs, bins=50, color='gray', alpha=0.6, density=True, label='H0: sin efecto')
ax.axvline(att, color='red', linewidth=2, linestyle='--', label=f'ATT observado=${att:,.0f}')
ax.axvline(-att, color='red', linewidth=2, linestyle='--', alpha=0.3)
ax.set_xlabel('Diferencia media (permutación)')
ax.set_ylabel('Densidad')
ax.set_title(f'Permutation Test (p={perm_pval:.4f})')
ax.legend()
plt.tight_layout()
plt.show()

## 2b. Lift por Quintil de Gasto Pre-Treatment

Agrupamos clientes en 5 quintiles de gasto pre-t0 (`monetary_total`) y estimamos ATT + significancia por quintil. Esto es comparable con la metodología existente (que usa deciles) pero con PSM en vez de matching naive.

In [ ]:
# ── Quintiles de gasto pre-treatment (desde snapshot) ────────────────────
# quintil_gasto viene precalculado en el snapshot (NTILE(5) OVER PARTITION BY t0)
if 'quintil_gasto' in df_cs.columns:
    QUINTIL_LABELS = {1: 'Q1 (bajo)', 2: 'Q2', 3: 'Q3', 4: 'Q4', 5: 'Q5 (alto)'}
    df_cs['gasto_quintil'] = df_cs['quintil_gasto'].map(QUINTIL_LABELS)
    print("Usando quintil_gasto del snapshot (calculado en Fase 1)")
else:
    df_cs['gasto_quintil'] = pd.qcut(
        df_cs['monetary_total'], 5, labels=['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)'],
        duplicates='drop'
    )
    print("WARN: quintil_gasto no encontrado en snapshot, calculado ad-hoc")

# Distribución por quintil
print(f"\nDistribución por quintil (en common support):")
for q in ['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)']:
    mask_q = df_cs['gasto_quintil'] == q
    n_q = mask_q.sum()
    treat_rate = df_cs.loc[mask_q, 'treatment'].mean()
    gasto_mean = df_cs.loc[mask_q, 'monetary_total'].mean()
    print(f"  {q}: n={n_q:,}, treatment={treat_rate:.1%}, gasto_pre_mean=${gasto_mean:,.0f}")

# Para cada quintil: ATT + test estadístico
quintil_results = []
gasto_quintiles_matched = df_cs['gasto_quintil'].values

for q in ['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)']:
    q_mask_treated = gasto_quintiles_matched[matched_treated_idx] == q
    
    y_t_q = Y_cs[matched_treated_idx[q_mask_treated]]
    y_c_q = Y_cs[matched_control_idx[q_mask_treated]]
    
    n_q = len(y_t_q)
    if n_q < 10:
        quintil_results.append({
            'quintil': q, 'n_pares': n_q,
            'gasto_treated': np.nan, 'gasto_control': np.nan,
            'att': np.nan, 'lift': np.nan,
            'ci_lo': np.nan, 'ci_hi': np.nan,
            't_stat': np.nan, 'p_ttest': np.nan,
            'w_stat': np.nan, 'p_wilcoxon': np.nan,
            'p_perm': np.nan, 'sig': ''
        })
        continue
    
    att_q = y_t_q.mean() - y_c_q.mean()
    lift_q = att_q / max(y_c_q.mean(), 1)
    
    # Sensitivity checks: t-test pareado y Wilcoxon
    t_q, p_t = stats.ttest_rel(y_t_q, y_c_q)
    
    try:
        w_q, p_w = stats.wilcoxon(y_t_q - y_c_q)
    except ValueError:
        w_q, p_w = np.nan, np.nan
    
    # Permutation test como test primario (no asume distribución)
    diffs = y_t_q - y_c_q
    obs_stat = diffs.mean()
    n_perm = 5000
    perm_count = 0
    for _ in range(n_perm):
        signs = np.random.choice([-1, 1], size=n_q)
        if abs((diffs * signs).mean()) >= abs(obs_stat):
            perm_count += 1
    perm_pval = perm_count / n_perm
    
    # Bootstrap CI
    boots_q = []
    for _ in range(1000):
        idx_b = np.random.choice(n_q, size=n_q, replace=True)
        boots_q.append(y_t_q[idx_b].mean() - y_c_q[idx_b].mean())
    ci_lo = np.percentile(boots_q, 2.5)
    ci_hi = np.percentile(boots_q, 97.5)
    
    # Usar permutation test como test primario (no asume distribución)
    sig = '***' if perm_pval < 0.01 else '**' if perm_pval < 0.05 else '*' if perm_pval < 0.10 else ''
    
    quintil_results.append({
        'quintil': q, 'n_pares': n_q,
        'gasto_treated': y_t_q.mean(), 'gasto_control': y_c_q.mean(),
        'att': att_q, 'lift': lift_q,
        'ci_lo': ci_lo, 'ci_hi': ci_hi,
        't_stat': t_q, 'p_ttest': p_t,
        'w_stat': w_q, 'p_wilcoxon': p_w,
        'p_perm': perm_pval, 'sig': sig
    })

qdf = pd.DataFrame(quintil_results)

print("\nATT POR QUINTIL DE GASTO PRE-TREATMENT")
print("="*120)
print(f"{'Quintil':<12} {'N':>6} {'Gasto T':>12} {'Gasto C':>12} {'ATT':>12} {'95% CI':>22} {'Lift':>8} {'p(perm)':>8} {'p(t)':>8} {'p(W)':>8} {'Sig':>4}")
print("-"*120)
for _, r in qdf.iterrows():
    if pd.isna(r['att']):
        print(f"{r['quintil']:<12} {r['n_pares']:>6} {'insuf. datos':>60}")
    else:
        ci_str = f"[{r['ci_lo']:>9,.0f}, {r['ci_hi']:>9,.0f}]"
        print(f"{r['quintil']:<12} {r['n_pares']:>6,} ${r['gasto_treated']:>10,.0f} ${r['gasto_control']:>10,.0f} ${r['att']:>10,.0f} {ci_str:>22} {r['lift']:>7.1%} {r['p_perm']:>8.4f} {r['p_ttest']:>8.4f} {r['p_wilcoxon']:>8.4f} {r['sig']:>4}")

print(f"\nSignificancia basada en permutation test (primario): *** p<0.01, ** p<0.05, * p<0.10")
print(f"p(t) y p(W) reportados como sensitivity checks")

In [ ]:
# ── Visualización quintiles ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

valid_q = qdf.dropna(subset=['att'])

# 1) ATT por quintil con CI
axes[0].bar(range(len(valid_q)), valid_q['att'],
            color=['green' if a > 0 else 'red' for a in valid_q['att']], alpha=0.7)
axes[0].errorbar(range(len(valid_q)), valid_q['att'],
                 yerr=[valid_q['att'] - valid_q['ci_lo'], valid_q['ci_hi'] - valid_q['att']],
                 fmt='none', color='black', capsize=4)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_xticks(range(len(valid_q)))
axes[0].set_xticklabels(valid_q['quintil'], rotation=15)
axes[0].set_ylabel('ATT ($)')
axes[0].set_title('ATT por Quintil de Gasto Pre (con 95% CI)')
# Marcar significativos
for i, (_, r) in enumerate(valid_q.iterrows()):
    if r['sig']:
        axes[0].text(i, r['att'] + (valid_q['att'].max() * 0.05), r['sig'],
                     ha='center', fontsize=12, fontweight='bold')

# 2) Gasto treated vs control por quintil
x = np.arange(len(valid_q))
w = 0.35
axes[1].bar(x - w/2, valid_q['gasto_treated'], w, label='Canjeó', color='steelblue', alpha=0.7)
axes[1].bar(x + w/2, valid_q['gasto_control'], w, label='No canjeó (matched)', color='gray', alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(valid_q['quintil'], rotation=15)
axes[1].set_ylabel('Gasto post-6m ($)')
axes[1].set_title('Gasto Post: Treated vs Control Matcheado')
axes[1].legend()

# 3) Lift % por quintil
colors = ['green' if l > 0 else 'red' for l in valid_q['lift']]
axes[2].bar(range(len(valid_q)), valid_q['lift'] * 100, color=colors, alpha=0.7)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].axhline(att_lift * 100, color='blue', linestyle='--', alpha=0.5, label=f'ATT global={att_lift:.1%}')
axes[2].set_xticks(range(len(valid_q)))
axes[2].set_xticklabels(valid_q['quintil'], rotation=15)
axes[2].set_ylabel('Lift (%)')
axes[2].set_title('Lift PSM por Quintil')
axes[2].legend()

plt.tight_layout()
plt.show()

# Resumen interpretativo
print("\nInterpretación:")
sig_q = valid_q[valid_q['sig'].str.len() > 0]
if len(sig_q) > 0:
    print(f"  Quintiles significativos: {', '.join(sig_q['quintil'].values)}")
else:
    print("  Ningún quintil es significativo (esperado con mock de 1000 clientes)")
print(f"  En producción con 500K+ clientes, se espera significancia en la mayoría de quintiles")

In [ ]:
# ── Lift por retailer (comparable con metodologia existente) ──────────────
# Calcular ATT por dominant_retailer
retailers_matched = df_cs['dominant_retailer'].values

retailer_lifts = []
for ret in ['STOREA', 'STOREB', 'STOREC', 'STORED', 'STOREE']:
    # Indices matched que pertenecen a este retailer (del treated)
    ret_mask = retailers_matched[matched_treated_idx] == ret
    if ret_mask.sum() < 10:
        retailer_lifts.append({'retailer': ret, 'n': ret_mask.sum(), 'att': np.nan, 'lift': np.nan})
        continue
    
    y_t = Y_cs[matched_treated_idx[ret_mask]]
    y_c = Y_cs[matched_control_idx[ret_mask]]
    att_r = y_t.mean() - y_c.mean()
    lift_r = att_r / max(y_c.mean(), 1)
    retailer_lifts.append({'retailer': ret, 'n': ret_mask.sum(), 'att': att_r, 'lift': lift_r})

lift_df = pd.DataFrame(retailer_lifts)

fig, ax = plt.subplots(figsize=(10, 5))
valid = lift_df.dropna(subset=['lift'])
colors = ['green' if l > 0 else 'red' for l in valid['lift']]
ax.bar(valid['retailer'], valid['lift'] * 100, color=colors, alpha=0.7)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Lift PSM (%)')
ax.set_title('Lift Incremental por Retailer (PSM)')
for i, row in valid.iterrows():
    ax.text(i, row['lift']*100 + 1, f"n={row['n']:.0f}", ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print("Lift por retailer:")
print(lift_df.round(3).to_string(index=False))

## 3. Uplift Modeling (T-Learner)

In [ ]:
# ── T-Learner: modelos separados para treated y control ──────────────────
# μ1(x) = E[Y | T=1, X=x]  (modelo para treated)
# μ0(x) = E[Y | T=0, X=x]  (modelo para control)
# CATE(x) = μ1(x) - μ0(x)  (efecto individual)
#
# Cross-fitted: 2-fold split to avoid in-sample prediction bias.
# Fold A trains μ1/μ0, predicts on Fold B (and vice versa).
# shuffle=False to respect temporal ordering.

X_all = X_psm.values

from sklearn.model_selection import KFold

kf = KFold(n_splits=2, shuffle=False)
mu1 = np.zeros(len(X_all))
mu0 = np.zeros(len(X_all))

for fold_train, fold_pred in kf.split(X_all):
    # Train on fold_train, predict on fold_pred
    mask_t1_train = T[fold_train] == 1
    mask_t0_train = T[fold_train] == 0

    m_t1 = GradientBoostingRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
    m_t0 = GradientBoostingRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
    m_t1.fit(X_all[fold_train[mask_t1_train]], Y[fold_train[mask_t1_train]])
    m_t0.fit(X_all[fold_train[mask_t0_train]], Y[fold_train[mask_t0_train]])

    mu1[fold_pred] = m_t1.predict(X_all[fold_pred])
    mu0[fold_pred] = m_t0.predict(X_all[fold_pred])

cate = mu1 - mu0

# Final models on all data (for downstream use in X-Learner and other notebooks)
model_t1 = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)
model_t0 = GradientBoostingRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, random_state=42
)
model_t1.fit(X_all[T == 1], Y[T == 1])
model_t0.fit(X_all[T == 0], Y[T == 0])

df['mu1'] = mu1
df['mu0'] = mu0
df['uplift'] = cate

print("T-Learner entrenado (cross-fitted, 2-fold)")
print(f"  μ1 final R² train: {model_t1.score(X_all[T==1], Y[T==1]):.4f}")
print(f"  μ0 final R² train: {model_t0.score(X_all[T==0], Y[T==0]):.4f}")
print(f"\nCate (uplift) distribution (cross-fitted):")
print(f"  mean:   ${cate.mean():>10,.0f}")
print(f"  median: ${np.median(cate):>10,.0f}")
print(f"  std:    ${cate.std():>10,.0f}")
print(f"  min:    ${cate.min():>10,.0f}")
print(f"  max:    ${cate.max():>10,.0f}")
print(f"  % positivo: {(cate > 0).mean():.1%}")

In [ ]:
# ── X-Learner: refinamiento del T-Learner (Cross-Fitted) ─────────────────
# Cross-fitted residuals: compute D1/D0 using held-out μ0/μ1 predictions
# to avoid fitting tau models on in-sample residuals.
#
# Uses the cross-fitted mu1/mu0 from the T-Learner cell above.

# Imputed treatment effects using CROSS-FITTED predictions (not in-sample)
D1 = Y[T == 1] - mu0[T == 1]  # treated: actual outcome - cross-fitted counterfactual
D0 = mu1[T == 0] - Y[T == 0]  # control: cross-fitted counterfactual - actual outcome

# Cross-fit the tau models too (2-fold, shuffle=False for temporal ordering)
cate_x = np.zeros(len(X_all))
idx_t1 = np.where(T == 1)[0]
idx_t0 = np.where(T == 0)[0]

kf_tau = KFold(n_splits=2, shuffle=False)

# tau1 cross-fitted predictions (on all data)
tau1_pred = np.zeros(len(X_all))
for tr, pr in kf_tau.split(idx_t1):
    m = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
    m.fit(X_all[idx_t1[tr]], D1[tr])
    tau1_pred += m.predict(X_all) / 2  # average over folds

# tau0 cross-fitted predictions (on all data)
tau0_pred = np.zeros(len(X_all))
for tr, pr in kf_tau.split(idx_t0):
    m = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
    m.fit(X_all[idx_t0[tr]], D0[tr])
    tau0_pred += m.predict(X_all) / 2  # average over folds

# Also train final tau models on all data (for downstream use)
tau1_model = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
tau0_model = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
tau1_model.fit(X_all[T == 1], D1)
tau0_model.fit(X_all[T == 0], D0)

# X-Learner CATE: weighted average using cross-fitted propensity
e = propensity  # cross-fitted propensity from earlier cell
cate_x = e * tau0_pred + (1 - e) * tau1_pred

df['uplift_x'] = cate_x

print("X-Learner entrenado (cross-fitted)")
print(f"\nCate X-Learner distribution:")
print(f"  mean:   ${cate_x.mean():>10,.0f}")
print(f"  median: ${np.median(cate_x):>10,.0f}")
print(f"  % positivo: {(cate_x > 0).mean():.1%}")
print(f"\nCorrelacion T-Learner vs X-Learner: {np.corrcoef(cate, cate_x)[0,1]:.4f}")

In [ ]:
# ── Visualización de Uplift ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribucion de CATE (T-Learner vs X-Learner)
axes[0].hist(cate, bins=50, alpha=0.6, label='T-Learner', color='steelblue')
axes[0].hist(cate_x, bins=50, alpha=0.6, label='X-Learner', color='coral')
axes[0].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Uplift ($)')
axes[0].set_title('Distribución de Uplift Individual')
axes[0].legend()

# Uplift vs Propensity Score
axes[1].scatter(propensity, cate_x, alpha=0.1, s=5, color='steelblue')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Propensity Score')
axes[1].set_ylabel('Uplift ($)')
axes[1].set_title('Uplift vs Propensity Score')

# Qini curve: uplift acumulado por decil
df_sorted = df.sort_values('uplift_x', ascending=False)
n = len(df_sorted)
pcts = np.arange(1, 11) / 10
cum_uplift = []
for p in pcts:
    top = df_sorted.head(int(n * p))
    treated_top = top[top['treatment'] == 1]['spending_post_6m'].mean()
    control_top = top[top['treatment'] == 0]['spending_post_6m'].mean()
    cum_uplift.append(treated_top - control_top)

axes[2].plot(pcts * 100, cum_uplift, 'bo-', linewidth=2)
axes[2].axhline(att, color='red', linestyle='--', alpha=0.5, label=f'ATT PSM=${att:,.0f}')
axes[2].set_xlabel('% población (top uplift)')
axes[2].set_ylabel('Diferencia gasto ($)')
axes[2].set_title('Uplift por Decil (ordenado por CATE)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. Revenue Decomposition

In [ ]:
# ── Revenue Decomposition: base + incremental ────────────────────────────
# Para cada treated: base = μ0(x), incremental = Y - μ0(x)
# Solo para treated (los que realmente canjearon)

df_treated = df[df['treatment'] == 1].copy()

df_treated['revenue_base'] = df_treated['mu0']         # lo que hubiera gastado SIN canjear
df_treated['revenue_incremental'] = df_treated['spending_post_6m'] - df_treated['mu0']
df_treated['revenue_total'] = df_treated['spending_post_6m']

# Resumen
total_base = df_treated['revenue_base'].sum()
total_incr = df_treated['revenue_incremental'].sum()
total_rev = df_treated['revenue_total'].sum()

print("REVENUE DECOMPOSITION (solo treated)")
print("="*60)
print(f"Revenue total:       ${total_rev:>14,.0f}  (100%)")
print(f"Revenue base:        ${total_base:>14,.0f}  ({total_base/max(total_rev,1)*100:.1f}%)")
print(f"Revenue incremental: ${total_incr:>14,.0f}  ({total_incr/max(total_rev,1)*100:.1f}%)")

# Por retailer
print(f"\nPor retailer:")
decomp_ret = df_treated.groupby('dominant_retailer').agg(
    n=('cust_id', 'count'),
    rev_total=('revenue_total', 'sum'),
    rev_base=('revenue_base', 'sum'),
    rev_incr=('revenue_incremental', 'sum'),
).reset_index()
decomp_ret['pct_incremental'] = decomp_ret['rev_incr'] / decomp_ret['rev_total'].clip(1) * 100
print(decomp_ret.round(0).to_string(index=False))

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(['Base', 'Incremental'], [total_base, total_incr],
            color=['gray', 'steelblue'])
axes[0].set_ylabel('Revenue ($)')
axes[0].set_title('Revenue Decomposition')

ret_plot = decomp_ret[decomp_ret['dominant_retailer'] != 'NINGUNO'].copy()
x = range(len(ret_plot))
axes[1].bar(x, ret_plot['rev_base'], label='Base', color='gray')
axes[1].bar(x, ret_plot['rev_incr'], bottom=ret_plot['rev_base'], label='Incremental', color='steelblue')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(ret_plot['dominant_retailer'], rotation=45)
axes[1].set_ylabel('Revenue ($)')
axes[1].set_title('Revenue Decomposition por Retailer')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Expected Value para Decision Engine

In [ ]:
# ── Expected Value = P(canje) × Uplift ──────────────────────────────────
# Esto alimenta al Decision Engine (Fase 2e) para priorización

df['expected_value'] = df['propensity_score'] * df['uplift_x']

# Prioridad basada en EV
ev_q60 = df['expected_value'].quantile(0.60)  # bottom 60% = baja
ev_q80 = df['expected_value'].quantile(0.80)  # top 20% = alta

df['prioridad'] = pd.cut(
    df['expected_value'],
    bins=[-np.inf, ev_q60, ev_q80, np.inf],
    labels=['Baja', 'Media', 'Alta']
)

print("EXPECTED VALUE = P(canje) × Uplift")
print("="*60)
print(f"\nDistribucion de EV:")
print(f"  mean:   ${df['expected_value'].mean():>10,.0f}")
print(f"  median: ${df['expected_value'].median():>10,.0f}")
print(f"  p60:    ${ev_q60:>10,.0f}")
print(f"  p80:    ${ev_q80:>10,.0f}")

print(f"\nPrioridad:")
for p in ['Alta', 'Media', 'Baja']:
    mask = df['prioridad'] == p
    print(f"  {p}: {mask.sum():,} ({mask.mean():.1%}) — EV mean=${df.loc[mask, 'expected_value'].mean():,.0f}")

# Validar: top 20% EV debería tener mayor tasa real de canje
print(f"\nValidacion: tasa de canje real por prioridad:")
for p in ['Alta', 'Media', 'Baja']:
    mask = df['prioridad'] == p
    tasa = df.loc[mask, 'treatment'].mean()
    print(f"  {p}: {tasa:.1%}")

In [ ]:
# ── Visualización de Expected Value ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# EV distribution by treatment
axes[0].hist(df.loc[df['treatment']==0, 'expected_value'], bins=50, alpha=0.6,
             label='No canjeó', color='gray', density=True)
axes[0].hist(df.loc[df['treatment']==1, 'expected_value'], bins=50, alpha=0.6,
             label='Canjeó', color='steelblue', density=True)
axes[0].set_xlabel('Expected Value ($)')
axes[0].set_title('Expected Value por Treatment')
axes[0].legend()

# Lift por decil de EV
df['ev_decile'] = pd.qcut(df['expected_value'], 10, labels=False, duplicates='drop')
decile_stats = df.groupby('ev_decile').agg(
    treatment_rate=('treatment', 'mean'),
    spending_mean=('spending_post_6m', 'mean'),
    uplift_mean=('uplift_x', 'mean'),
    n=('cust_id', 'count'),
).reset_index()

ax2 = axes[1]
ax2.bar(decile_stats['ev_decile'], decile_stats['treatment_rate'] * 100,
        color='steelblue', alpha=0.7, label='Tasa canje (%)')
ax2_twin = ax2.twinx()
ax2_twin.plot(decile_stats['ev_decile'], decile_stats['uplift_mean'],
              'ro-', linewidth=2, label='Uplift medio ($)')
ax2.set_xlabel('Decil de Expected Value')
ax2.set_ylabel('Tasa de canje (%)')
ax2_twin.set_ylabel('Uplift medio ($)')
ax2.set_title('Tasa de Canje y Uplift por Decil de EV')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()

print("Deciles de Expected Value:")
print(decile_stats.round(2).to_string(index=False))

## 6. Comparación con Metodología Existente (Lift por Decil)

In [ ]:
# ── Comparar con metodología existente: lift por quintil de gasto previo ──
# Producción usa deciles. Nosotros usamos quintiles (del snapshot, Fase 1)

if 'quintil_gasto' in df.columns:
    df['gasto_quintil_global'] = df['quintil_gasto'].map(
        {1: 'Q1 (bajo)', 2: 'Q2', 3: 'Q3', 4: 'Q4', 5: 'Q5 (alto)'}
    )
else:
    df['gasto_quintil_global'] = pd.qcut(
        df['monetary_total'], 5, labels=['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)'],
        duplicates='drop'
    )

lift_por_quintil = []
for q in ['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)']:
    mask_q = df['gasto_quintil_global'] == q
    treated_q = df.loc[mask_q & (df['treatment'] == 1), 'spending_post_6m']
    control_q = df.loc[mask_q & (df['treatment'] == 0), 'spending_post_6m']
    if len(treated_q) < 5 or len(control_q) < 5:
        continue
    
    lift_q = (treated_q.mean() - control_q.mean()) / max(control_q.mean(), 1)
    # Uplift model estimate por quintil
    uplift_q = df.loc[mask_q, 'uplift_x'].mean()
    base_q = df.loc[mask_q, 'mu0'].mean()
    lift_uplift_q = uplift_q / max(base_q, 1)
    
    lift_por_quintil.append({
        'quintil': q,
        'n_treated': len(treated_q),
        'n_control': len(control_q),
        'gasto_treated': treated_q.mean(),
        'gasto_control': control_q.mean(),
        'lift_naive': lift_q,
        'lift_uplift_model': lift_uplift_q,
    })

comp_df = pd.DataFrame(lift_por_quintil)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comp_df))
w = 0.35
ax.bar(x - w/2, comp_df['lift_naive'] * 100, w, label='Método existente (naive/quintil)', color='gray', alpha=0.7)
ax.bar(x + w/2, comp_df['lift_uplift_model'] * 100, w, label='Uplift model (X-Learner)', color='steelblue', alpha=0.7)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Quintil de Gasto Previo')
ax.set_ylabel('Lift (%)')
ax.set_xticks(x)
ax.set_xticklabels(comp_df['quintil'])
ax.set_title('Lift por Quintil: Método Existente vs Uplift Model')
ax.legend()
plt.tight_layout()
plt.show()

print("Comparación por quintil:")
print(comp_df.round(3).to_string(index=False))

## 6b. Metodología GitLab (Baseline de Control de Gestión)

La metodología que usa actualmente el equipo de Control de Gestión en Falabella (implementada en los scheduled workflows de GitLab) para medir incrementalidad de canjes. Se incluye aquí como **baseline de comparación** contra PSM y el Uplift Model.

### Diferencias clave con nuestro enfoque:
- **Sin matching:** Compara CANJEADOR vs POTENCIAL directamente (no controla confounders)
- **Survivorship bias:** Solo incluye clientes con gasto > 0 en pre Y post
- **Outlier filter:** Remueve top 1% en pre y post
- **Resultado esperado:** Sobreestima el lift porque los canjeadores son inherentemente clientes más activos

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# METODOLOGÍA GITLAB — Baseline de Control de Gestión
# Replica la lógica de los scheduled workflows de GitLab (producción actual)
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import ttest_ind

print("=" * 80)
print("METODOLOGÍA GITLAB (BASELINE)")
print("=" * 80)

# ── Paso 1: Clasificar clientes ──────────────────────────────────────────────
df_gl = df.copy()
df_gl["cliente_tipo"] = np.where(
    df_gl["treatment"] == 1, "CANJEADOR",
    np.where(df_gl["stock_points_at_t0"] >= 1000, "POTENCIAL", "ACUMULADOR")
)

print("\n  Distribución de tipos de cliente:")
for tipo in ["CANJEADOR", "POTENCIAL", "ACUMULADOR"]:
    n_t = (df_gl["cliente_tipo"] == tipo).sum()
    print(f"    {tipo:15s}: {n_t:>6,} ({n_t/len(df_gl)*100:.1f}%)")

# ── Paso 2: Solo CANJEADOR vs POTENCIAL ──────────────────────────────────────
df_gl = df_gl[df_gl["cliente_tipo"].isin(["CANJEADOR", "POTENCIAL"])].copy()

# ── Paso 3: Survivorship filter ──────────────────────────────────────────────
spending_col = "spending_post_6m" if "spending_post_6m" in df_gl.columns else "revenue_post_12m"
n_before = len(df_gl)
df_gl = df_gl[(df_gl["monetary_total"] > 0) & (df_gl[spending_col] > 0)]
pct_kept = len(df_gl) / n_before * 100 if n_before > 0 else 0
print(f"\n  Filtro survivorship: {n_before:,} → {len(df_gl):,} ({pct_kept:.0f}%)")
if pct_kept < 70:
    print(f"  ⚠ WARNING: Se descartó {100 - pct_kept:.0f}%. Sesgo de supervivencia significativo.")

# ── Paso 4: Outlier filter P99 ───────────────────────────────────────────────
p99_pre = df_gl["monetary_total"].quantile(0.99)
p99_post = df_gl[spending_col].quantile(0.99)
n_before = len(df_gl)
df_gl = df_gl[(df_gl["monetary_total"] <= p99_pre) & (df_gl[spending_col] <= p99_post)]
print(f"  Filtro outliers P99: {n_before:,} → {len(df_gl):,}")

# ── Paso 5: Quintiles ────────────────────────────────────────────────────────
QLABELS = {1: "Q1 (bajo)", 2: "Q2", 3: "Q3", 4: "Q4", 5: "Q5 (alto)"}
if "quintil_gasto" in df_gl.columns:
    df_gl["gasto_quintil"] = df_gl["quintil_gasto"].map(QLABELS)
else:
    df_gl["gasto_quintil"] = pd.qcut(
        df_gl["monetary_total"], 5, labels=list(QLABELS.values()), duplicates="drop")

# ── Paso 6: Lift por quintil ─────────────────────────────────────────────────
print(f"\n  {'Quintil':>18s} | {'N Canj':>7s} | {'N Pot':>7s} | {'Post Canj':>12s} | {'Post Pot':>12s} | {'Lift':>8s} | {'p-value':>10s}")
print("  " + "-" * 85)

gitlab_results = []
for q in ["Q1 (bajo)", "Q2", "Q3", "Q4", "Q5 (alto)"]:
    mask_q = df_gl["gasto_quintil"] == q
    canj = df_gl.loc[mask_q & (df_gl["cliente_tipo"] == "CANJEADOR"), spending_col]
    pot = df_gl.loc[mask_q & (df_gl["cliente_tipo"] == "POTENCIAL"), spending_col]
    if len(canj) < 5 or len(pot) < 5:
        continue
    lift = (canj.mean() - pot.mean()) / pot.mean() if pot.mean() > 0 else 0
    _, pval = ttest_ind(canj, pot)
    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else "ns"
    gitlab_results.append({"quintil": q, "n_canj": len(canj), "n_pot": len(pot),
        "post_canj": canj.mean(), "post_pot": pot.mean(), "lift": lift, "pval": pval})
    print(f"  {q:>18s} | {len(canj):>7,} | {len(pot):>7,} | ${canj.mean():>11,.0f} | ${pot.mean():>11,.0f} | {lift:>+7.1%} | {pval:>10.4f} {sig}")

if gitlab_results:
    total_cj = sum(r["post_canj"] * r["n_canj"] for r in gitlab_results)
    n_cj = sum(r["n_canj"] for r in gitlab_results)
    total_pt = sum(r["post_pot"] * r["n_pot"] for r in gitlab_results)
    n_pt = sum(r["n_pot"] for r in gitlab_results)
    if n_pt > 0 and n_cj > 0:
        lift_g = ((total_cj/n_cj) - (total_pt/n_pt)) / (total_pt/n_pt)
        print(f"\n  LIFT GLOBAL GitLab: {lift_g:+.1%}")

print("""
  COMPARACIÓN:
  - GitLab: sin matching, survivorship bias → sobreestima lift
  - PSM: matching por propensity, sin survivorship filter → más conservador
  - Uplift Model: efecto individual (CATE) → permite priorizar por impacto causal
""")

## 6b. Comparación Directa: Metodología GitLab vs PSM

Replicamos la metodología exacta de producción (GitLab) y comparamos los lifts con nuestra metodología PSM, quintil por quintil.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# METODO GITLAB (producción)
# ══════════════════════════════════════════════════════════════════════════
# Replica exacta:
# 1. Canjeador: canjeó en periodo SEL (y >= 1)
# 2. Canjeador Potencial: ≥1000 pts, NO canjeó
# 3. Excluir: GASTO_PREV <= 0 o GASTO_POST <= 0 (survivorship filter)
# 4. Excluir: top 1% gasto en PREV y POST (outlier removal)
# 5. Deciles de GASTO_PREV
# 6. Lift = (gasto_prom_canjeador - gasto_prom_potencial) / gasto_prom_potencial

df_gitlab = df.copy()

# --- Clasificar clientes ---
# Canjeador: canjeó post-t0
# Canjeador Potencial: ≥1000 pts y NO canjeó
# Acumulador No Potencial: <1000 pts y no canjeó (excluido del análisis)
df_gitlab['cliente_estudio'] = np.where(
    df_gitlab['treatment'] == 1, '3.CANJEADOR',
    np.where(
        df_gitlab['stock_points_at_t0'] >= 1000, '2.CANJEADOR POTENCIAL',
        '1.ACUMULADOR NO POTENCIAL'
    )
)

# Solo canjeadores y potenciales
df_gl = df_gitlab[df_gitlab['cliente_estudio'].isin(['3.CANJEADOR', '2.CANJEADOR POTENCIAL'])].copy()

# --- Filtro survivorship: GASTO_PREV > 0 AND GASTO_POST > 0 ---
# NOTA: El filtro GASTO_POST > 0 del método GitLab introduce sesgo de supervivencia:
# solo compara clientes activos post-tratamiento, inflando el lift estimado.
df_gl = df_gl[(df_gl['monetary_total'] > 0) & (df_gl['spending_post_6m'] > 0)]

# --- Excluir top 1% outliers en PREV y POST ---
p99_prev = df_gl['monetary_total'].quantile(0.99)
p99_post = df_gl['spending_post_6m'].quantile(0.99)
df_gl = df_gl[(df_gl['monetary_total'] <= p99_prev) & (df_gl['spending_post_6m'] <= p99_post)]

# --- Deciles de GASTO_PREV ---
df_gl['gasto_decil'] = pd.qcut(df_gl['monetary_total'], 10, labels=False, duplicates='drop')

# --- Quintiles (agrupar deciles: 0-1=Q1, 2-3=Q2, etc.) ---
df_gl['gasto_quintil_gl'] = pd.cut(
    df_gl['gasto_decil'], bins=[-1, 1, 3, 5, 7, 9],
    labels=['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)']
)

print("METODO GITLAB — Filtros aplicados:")
print(f"  Canjeadores: {(df_gl['cliente_estudio']=='3.CANJEADOR').sum():,}")
print(f"  Canjeadores Potenciales: {(df_gl['cliente_estudio']=='2.CANJEADOR POTENCIAL').sum():,}")
print(f"  Excluidos (acumuladores no potencial): {(df_gitlab['cliente_estudio']=='1.ACUMULADOR NO POTENCIAL').sum():,}")
print(f"  Excluidos (survivorship GASTO=0): filtrado")
print(f"  Excluidos (top 1% outliers): p99_prev=${p99_prev:,.0f}, p99_post=${p99_post:,.0f}")
print(f"  Total para análisis: {len(df_gl):,}")

In [ ]:
# ── Calcular lift GitLab por quintil ──────────────────────────────────────
gitlab_lifts = []
for q in ['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)']:
    mask_q = df_gl['gasto_quintil_gl'] == q
    canjeadores = df_gl.loc[mask_q & (df_gl['cliente_estudio'] == '3.CANJEADOR')]
    potenciales = df_gl.loc[mask_q & (df_gl['cliente_estudio'] == '2.CANJEADOR POTENCIAL')]
    
    if len(canjeadores) < 5 or len(potenciales) < 5:
        gitlab_lifts.append({'quintil': q, 'n_canj': len(canjeadores), 'n_pot': len(potenciales),
                             'gasto_canj': np.nan, 'gasto_pot': np.nan, 'lift_gitlab': np.nan})
        continue
    
    gasto_canj = canjeadores['spending_post_6m'].mean()
    gasto_pot = potenciales['spending_post_6m'].mean()
    lift_gl = (gasto_canj - gasto_pot) / gasto_pot
    
    gitlab_lifts.append({
        'quintil': q, 'n_canj': len(canjeadores), 'n_pot': len(potenciales),
        'gasto_canj': gasto_canj, 'gasto_pot': gasto_pot, 'lift_gitlab': lift_gl
    })

gl_df = pd.DataFrame(gitlab_lifts)

# Lift global GitLab (ponderado por gasto canjeador, como en porcentaje_lift.sql)
gl_valid = gl_df.dropna(subset=['lift_gitlab'])
if len(gl_valid) > 0:
    gl_valid = gl_valid.copy()
    gl_valid['lift_gasto'] = gl_valid['lift_gitlab'] * gl_valid['gasto_canj'] * gl_valid['n_canj']
    gasto_total_canj = (gl_valid['gasto_canj'] * gl_valid['n_canj']).sum()
    lift_global_gitlab = gl_valid['lift_gasto'].sum() / gasto_total_canj * 100
else:
    lift_global_gitlab = np.nan

print("LIFT METODO GITLAB (por quintil)")
print("="*90)
print(f"{'Quintil':<12} {'N canj':>8} {'N pot':>8} {'Gasto canj':>14} {'Gasto pot':>14} {'Lift':>8}")
print("-"*90)
for _, r in gl_df.iterrows():
    if pd.isna(r['lift_gitlab']):
        print(f"{r['quintil']:<12} {r['n_canj']:>8} {r['n_pot']:>8} {'insuf':>30}")
    else:
        print(f"{r['quintil']:<12} {r['n_canj']:>8,} {r['n_pot']:>8,} ${r['gasto_canj']:>12,.0f} ${r['gasto_pot']:>12,.0f} {r['lift_gitlab']:>7.1%}")

print(f"\nLift global GitLab (ponderado): {lift_global_gitlab:.1f}%")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# METODO PSM — MATCHING DENTRO DE CADA QUINTIL DE GASTO PRE
# ══════════════════════════════════════════════════════════════════════════
# Mismos filtros que GitLab (survivorship + outliers).
# Mejora: matching por propensity score DENTRO de cada quintil de gasto pre,
# asegurando que treated y control tienen nivel de gasto similar Y
# características observables similares.

df_psm2 = df_gitlab[df_gitlab['cliente_estudio'].isin(['3.CANJEADOR', '2.CANJEADOR POTENCIAL'])].copy()
df_psm2 = df_psm2[(df_psm2['monetary_total'] > 0) & (df_psm2['spending_post_6m'] > 0)]
df_psm2 = df_psm2[(df_psm2['monetary_total'] <= p99_prev) & (df_psm2['spending_post_6m'] <= p99_post)]

T2 = (df_psm2['cliente_estudio'] == '3.CANJEADOR').astype(int).values
Y2 = df_psm2['spending_post_6m'].values
X2_scaled = scaler.transform(df_psm2[PSM_FEATURES].fillna(0))
ps2 = ps_model.predict_proba(X2_scaled)[:, 1]

# Quintiles (same bins as GitLab)
df_psm2['gasto_decil'] = pd.qcut(df_psm2['monetary_total'], 10, labels=False, duplicates='drop')
df_psm2['gasto_quintil_psm'] = pd.cut(
    df_psm2['gasto_decil'], bins=[-1, 1, 3, 5, 7, 9],
    labels=['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)']
)

# ── Match WITHIN each quintile ──
m_t2_all = []
m_c2_all = []

for q in ['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)']:
    q_mask = df_psm2['gasto_quintil_psm'].values == q
    idx_t_q = np.where((T2 == 1) & q_mask)[0]
    idx_c_q = np.where((T2 == 0) & q_mask)[0]
    
    if len(idx_t_q) < 5 or len(idx_c_q) < 5:
        continue
    
    nn_q = NearestNeighbors(n_neighbors=1, metric='euclidean')
    nn_q.fit(ps2[idx_c_q].reshape(-1, 1))
    dist_q, neigh_q = nn_q.kneighbors(ps2[idx_t_q].reshape(-1, 1))
    matched_c_q = idx_c_q[neigh_q.flatten()]
    
    good_q = dist_q.flatten() <= CALIPER
    m_t2_all.extend(idx_t_q[good_q])
    m_c2_all.extend(matched_c_q[good_q])

m_t2 = np.array(m_t2_all)
m_c2 = np.array(m_c2_all)

# Lift PSM por quintil (within-quintile matched)
psm_lifts = []
quintiles_psm = df_psm2['gasto_quintil_psm'].values

for q in ['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)']:
    q_mask = quintiles_psm[m_t2] == q
    y_t = Y2[m_t2[q_mask]]
    y_c = Y2[m_c2[q_mask]]
    
    n_q = len(y_t)
    if n_q < 5:
        psm_lifts.append({'quintil': q, 'n_pares': n_q, 'att_psm': np.nan, 'lift_psm': np.nan})
        continue
    
    att_q = y_t.mean() - y_c.mean()
    lift_q = att_q / max(y_c.mean(), 1)
    t_q, p_q = stats.ttest_rel(y_t, y_c)
    
    psm_lifts.append({
        'quintil': q, 'n_pares': n_q,
        'gasto_t_psm': y_t.mean(), 'gasto_c_psm': y_c.mean(),
        'att_psm': att_q, 'lift_psm': lift_q,
        'p_value': p_q
    })

psm2_df = pd.DataFrame(psm_lifts)

# ATT global PSM
att_global_psm2 = Y2[m_t2].mean() - Y2[m_c2].mean()
lift_global_psm2 = att_global_psm2 / max(Y2[m_c2].mean(), 1) * 100

print(f"PSM re-estimado (matching DENTRO de cada quintil de gasto pre):")
print(f"  Pares matcheados: {len(m_t2):,}")
print(f"  Lift global PSM: {lift_global_psm2:.1f}%")

# Verify gasto pre balance
print(f"
  Balance gasto pre (should be similar within quintile):")
for q in ['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)']:
    q_mask = quintiles_psm[m_t2] == q
    if q_mask.sum() < 5:
        continue
    pre_t = df_psm2.iloc[m_t2[q_mask]]['monetary_total'].mean()
    pre_c = df_psm2.iloc[m_c2[q_mask]]['monetary_total'].mean()
    print(f"    {q}: treated=${pre_t:,.0f}, control=${pre_c:,.0f}, diff=${pre_t-pre_c:+,.0f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# COMPARACION DIRECTA: GitLab vs PSM (misma población, mismo filtro)
# ══════════════════════════════════════════════════════════════════════════
comp = gl_df[['quintil', 'n_canj', 'n_pot', 'gasto_canj', 'gasto_pot', 'lift_gitlab']].merge(
    psm2_df[['quintil', 'n_pares', 'gasto_t_psm', 'gasto_c_psm', 'lift_psm', 'p_value']],
    on='quintil', how='outer'
)

comp['diff_pp'] = (comp['lift_gitlab'] - comp['lift_psm']) * 100  # diferencia en pp

print("COMPARACION DIRECTA: GITLAB vs PSM")
print("="*120)
print(f"{'Quintil':<12} │{'── GitLab ──────────────────':^30}│{'── PSM ─────────────────────────':^34}│ {'Diff':>6}")
print(f"{'':12} │ {'N canj':>6} {'N pot':>6} {'Lift':>7} │ {'N par':>6} {'Gasto T':>10} {'Gasto C':>10} {'Lift':>7} {'p-val':>7} │ {'(pp)':>6}")
print("-"*120)

for _, r in comp.iterrows():
    gl_str = f" {r['n_canj']:>6.0f} {r['n_pot']:>6.0f} {r['lift_gitlab']:>6.1%}" if not pd.isna(r['lift_gitlab']) else f" {'—':>6} {'—':>6} {'—':>7}"
    psm_str = f" {r['n_pares']:>6.0f} ${r['gasto_t_psm']:>8,.0f} ${r['gasto_c_psm']:>8,.0f} {r['lift_psm']:>6.1%} {r['p_value']:>7.4f}" if not pd.isna(r['lift_psm']) else f" {'—':>40}"
    diff_str = f" {r['diff_pp']:>+5.1f}" if not pd.isna(r['diff_pp']) else f" {'—':>5}"
    print(f"{r['quintil']:<12} │{gl_str} │{psm_str} │{diff_str}")

print("-"*120)
print(f"{'GLOBAL':<12} │ {'':>14} {lift_global_gitlab:>6.1f}% │ {'':>30} {lift_global_psm2:>6.1f}% {'':>8}│ {lift_global_gitlab - lift_global_psm2:>+5.1f}")

# ── Gráfico comparativo ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

valid_comp = comp.dropna(subset=['lift_gitlab', 'lift_psm'])
x = np.arange(len(valid_comp))
w = 0.35

# Lift por quintil
axes[0].bar(x - w/2, valid_comp['lift_gitlab'] * 100, w, label='GitLab (naive/decil)', color='#e74c3c', alpha=0.7)
axes[0].bar(x + w/2, valid_comp['lift_psm'] * 100, w, label='PSM (propensity match)', color='#2980b9', alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(valid_comp['quintil'])
axes[0].set_ylabel('Lift (%)')
axes[0].set_title('Lift por Quintil: GitLab vs PSM')
axes[0].legend()

# Diferencia (GitLab - PSM) en pp
axes[1].bar(x, valid_comp['diff_pp'], color=['#e74c3c' if d > 0 else '#2980b9' for d in valid_comp['diff_pp']], alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(valid_comp['quintil'])
axes[1].set_ylabel('GitLab - PSM (pp)')
axes[1].set_title('Sesgo del método GitLab vs PSM (rojo = GitLab sobreestima)')

plt.tight_layout()
plt.show()

print("\nInterpretación:")
print(f"  Lift global GitLab: {lift_global_gitlab:>+.1f}%")
print(f"  Lift global PSM:    {lift_global_psm2:>+.1f}%")
overest = lift_global_gitlab - lift_global_psm2
if abs(overest) > 1:
    direction = "sobreestima" if overest > 0 else "subestima"
    print(f"  → GitLab {direction} el lift en {abs(overest):.1f}pp vs PSM")
else:
    print(f"  → Diferencia marginal ({overest:+.1f}pp)")
print(f"  → Con datos reales, la diferencia será más pronunciada por el sesgo de selección")

In [ ]:
# ── Merge ambos lifts al dataframe principal ─────────────────────────────
# Cada cliente recibe el lift de su quintil (ambos métodos) + uplift individual

# Asignar quintil al df principal (si no existe)
if 'quintil_gasto' in df.columns:
    df['gasto_quintil_label'] = df['quintil_gasto'].map(
        {1: 'Q1 (bajo)', 2: 'Q2', 3: 'Q3', 4: 'Q4', 5: 'Q5 (alto)'}
    )
else:
    df['gasto_quintil_label'] = pd.qcut(
        df['monetary_total'], 5, labels=['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)'],
        duplicates='drop'
    )

# Merge lift GitLab por quintil
lift_gl_map = gl_df.set_index('quintil')['lift_gitlab'].to_dict()
df['lift_gitlab_quintil'] = df['gasto_quintil_label'].map(lift_gl_map)

# Merge lift PSM por quintil
lift_psm_map = psm2_df.set_index('quintil')['lift_psm'].to_dict()
df['lift_psm_quintil'] = df['gasto_quintil_label'].map(lift_psm_map)

# Resumen: 3 columnas de incrementalidad disponibles
print("COLUMNAS DE INCREMENTALIDAD EN EL MODELO")
print("="*70)
print(f"{'Columna':<25} {'Nivel':<15} {'Método':<20} {'Valores'}")
print("-"*70)
print(f"{'lift_gitlab_quintil':<25} {'por quintil':<15} {'GitLab (naive)':<20} {df['lift_gitlab_quintil'].describe()[['mean','min','max']].to_dict()}")
print(f"{'lift_psm_quintil':<25} {'por quintil':<15} {'PSM (causal)':<20} {df['lift_psm_quintil'].describe()[['mean','min','max']].to_dict()}")
print(f"{'uplift_x':<25} {'individual':<15} {'X-Learner':<20} {df['uplift_x'].describe()[['mean','min','max']].to_dict()}")

print(f"\nMuestra (5 clientes):")
cols_show = ['cust_id', 't0', 'gasto_quintil_label', 'treatment',
             'lift_gitlab_quintil', 'lift_psm_quintil', 'uplift_x', 'expected_value', 'prioridad']
print(df[cols_show].sample(5, random_state=42).to_string(index=False))

print(f"\nTotal columnas incrementalidad: 3 (GitLab por quintil, PSM por quintil, X-Learner individual)")
print(f"El Decision Engine (2e) puede usar cualquiera o combinar.")

## 7. Resumen

In [ ]:
# ── Resumen ejecutivo ────────────────────────────────────────────────────
print("="*80)
print("RESUMEN FASE 2d: INCREMENTALIDAD")
print("="*80)

print(f"\n1. PROPENSITY SCORE MATCHING")
print(f"   Propensity model AUC: {auc_ps:.4f}")
print(f"   Pares matcheados: {len(matched_treated_idx):,}")
print(f"   Balance post-matching: {n_balanced}/{len(PSM_FEATURES)} covariables con |SMD|≤0.1")
print(f"   ATT (causal): ${att:,.0f} [{att_ci_lo:,.0f}, {att_ci_hi:,.0f}]")
print(f"   Lift PSM: {att_lift:.1%}")

print(f"\n2. UPLIFT MODELING")
print(f"   T-Learner CATE medio: ${cate.mean():,.0f}")
print(f"   X-Learner CATE medio: ${cate_x.mean():,.0f}")
print(f"   Correlacion T vs X:   {np.corrcoef(cate, cate_x)[0,1]:.4f}")
print(f"   % con uplift > 0:     {(cate_x > 0).mean():.1%}")

print(f"\n3. REVENUE DECOMPOSITION")
print(f"   Revenue base:        ${total_base:>14,.0f} ({total_base/max(total_rev,1)*100:.1f}%)")
print(f"   Revenue incremental: ${total_incr:>14,.0f} ({total_incr/max(total_rev,1)*100:.1f}%)")

print(f"\n4. COMPARACION VS METODO EXISTENTE")
print(f"   Lift naive (sin matching):   {naive_lift:.1%}")
print(f"   Lift PSM (causal):           {att_lift:.1%}")
print(f"   Diferencia = sesgo estimado: {abs(naive_lift - att_lift)*100:.1f}pp")

print(f"\n5. OUTPUT PARA DECISION ENGINE (Fase 2e)")
print(f"   Variables generadas por cliente:")
print(f"     - propensity_score: P(canje|X)")
print(f"     - uplift_x: CATE individual (X-Learner)")
print(f"     - expected_value: P(canje) × Uplift")
print(f"     - prioridad: Alta/Media/Baja")

print(f"\nNota: Con datos mock los uplift son orientativos.")
print(f"En produccion el PSM corregira sesgos reales de seleccion.")

---
## Next Steps

- **Fase 2e**: Decision Engine integra propensity, uplift, clusters
- En producción: calcular `top_up_spending = total_amount(txn asociada) - redeem_amount` vía `associated_transaction_id`
- Evaluar si X-Learner mejora significativamente sobre T-Learner (con datos reales)
- Considerar doubly-robust estimator para mayor robustez